# 6장 · 연쇄법칙 🔗 (핵심 — 역전파의 심장)

웹 챕터 **"연쇄법칙"** 과 짝을 이루는 Python 실습입니다.
합성함수를 SymPy 로 미분해 연쇄법칙과 맞춰보고, 국소 미분의 곱이 전체 미분과 같음을
수치로 확인합니다. (역전파의 축소판!)

준비물: `pip install -r requirements.txt`

> 🤖 **AI 튜터**: 아래 셀을 실행하면 웹앱처럼 화면 오른쪽 아래에 🤖 버튼이 생깁니다.
> 버튼을 누르면 채팅창이 열리고, 어디를 스크롤해도 그 자리에 떠 있어요. 막히면 언제든 질문하세요.
> (로컬 Ollama 필요. 꺼져 있으면 켜는 방법이 안내되고, 튜터 없이도 학습에는 지장 없습니다.)

In [ ]:
# 🤖 AI 튜터 — 실행하면 오른쪽 아래에 🤖 버튼이 떠요 (웹앱과 같은 로컬 AI, 클릭해서 열기)
from tutor import chat

chat('ch6')

## 1. 합성함수와 연쇄법칙

$$\big(f(g(x))\big)' = f'(g(x))\cdot g'(x) \quad(\text{겉미분} \times \text{속미분})$$

속 g(x)=2x+1, 겉 f(u)=u² 로 합성한 y=(2x+1)² 을 미분해 봅니다.

In [ ]:
import sympy as sp

x, u = sp.symbols('x u')
g = 2*x + 1        # 속함수
f = u**2           # 겉함수 (변수 u)

# 연쇄법칙을 손으로: f'(g)·g'
by_chain = f.diff(u).subs(u, g) * g.diff(x)
# SymPy 가 직접 미분
by_sympy = sp.diff(f.subs(u, g), x)

print('연쇄법칙(손) :', sp.expand(by_chain))    # 8x + 4 = 4(2x+1)
print('SymPy       :', sp.expand(by_sympy))
print('같은가?      :', sp.simplify(by_chain - by_sympy) == 0)

## 2. 또 다른 예: (x²+1)³

속 g=x²+1, 겉 f=u³. 겉미분 3u² (속 그대로) × 속미분 2x.

In [ ]:
g2 = x**2 + 1
f2 = u**3
by_chain2 = f2.diff(u).subs(u, g2) * g2.diff(x)
by_sympy2 = sp.diff(f2.subs(u, g2), x)
print('연쇄법칙(손) :', sp.factor(by_chain2))    # 6x(x²+1)²
print('SymPy       :', sp.factor(by_sympy2))
print('x=1 에서 값  :', by_sympy2.subs(x, 1))     # 24

## 3. 국소 미분의 '곱' = 전체 미분 (수치 = 역전파 축소판)

x → u=g(x) → y=f(u) 사슬에서 각 단계의 국소 변화율을 수치로 구해 곱하면,
전체 변화율 dy/dx 와 같아집니다. 이것이 역전파가 하는 일의 축소판입니다.

In [ ]:
def gg(x):
    return 2*x + 1        # 속함수
def ff(u):
    return u**2           # 겉함수

def d(func, a, h=1e-6):
    return (func(a + h) - func(a - h)) / (2*h)   # 수치 미분

x0 = 1.0
u0 = gg(x0)
du_dx = d(gg, x0)         # 국소: du/dx = g'(x)
dy_du = d(ff, u0)         # 국소: dy/du = f'(u)
chain = dy_du * du_dx     # 곱 (연쇄법칙)

whole = d(lambda t: ff(gg(t)), x0)   # 전체 dy/dx 직접
print(f'du/dx = {du_dx:.4f},  dy/du = {dy_du:.4f}')
print(f'곱(연쇄법칙) = {chain:.4f}')
print(f'전체 dy/dx  = {whole:.4f}')
print('거의 같은가?:', abs(chain - whole) < 1e-4)

## 4. 직접 풀어보기 ✏️

연쇄법칙으로 **y=(2x+1)² 의 x=1 에서의 미분값**을 구해 보세요.
(웹 문제와 같습니다. 답은 12!)

In [ ]:
# TODO: (2x+1)² 을 미분해 x=1 을 대입해 answer 에 담으세요.
# 힌트: sp.diff((2*x+1)**2, x).subs(x, 1)

answer = sp.diff((2*x + 1)**2, x).subs(x, 1)
print('정답:', answer)
assert answer == 12, '다시! 도함수 4(2x+1), x=1 이면 12. (겉미분 6 × 속미분 2)'
print('✅ 잘했어요! f\'(u)·g\'(x) = 6·2 = 12, 겉미분 × 속미분.')

---
다음 장(7장): AI 활성화 함수의 미분, 곧 **지수(eˣ)·로그(ln)·시그모이드(σ)** 를 배웁니다.
특히 시그모이드는 σ' = σ(1−σ) 라는 예쁜 성질이 있어요. 🙂